# Feature Engineering: Polynomial Features and Mathematical Transformations

This notebook uses the **UCI Wine Quality (Red Wine)** tabular dataset to demonstrate:
- initial data cleaning and a baseline Linear Regression model
- polynomial features and interaction terms
- a custom `log1p` mathematical transformation for skewed variables
- retraining and comparison using MAE, RMSE and R²
- a short interpretability discussion


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42


## 1. Load the UCI dataset


In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
df = pd.read_csv(url, sep=';')
print('Shape:', df.shape)
display(df.head())


## 2. Initial data cleaning and EDA


In [ ]:
print(df.info())
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nSummary statistics:')
display(df.describe().T)


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['quality'], discrete=True)
plt.title('Distribution of Wine Quality')
plt.xlabel('Quality')
plt.show()

skewness = df.drop(columns=['quality']).skew().sort_values(ascending=False)
display(skewness.to_frame('skewness'))


## 3. Train/test split

The split is performed before model fitting so that the test set remains unseen during training.

In [ ]:
X = df.drop(columns=['quality']).copy()
y = df['quality'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## 4. Baseline model
A simple Linear Regression model is used as the performance benchmark.

In [ ]:
baseline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

def metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred, squared=False),
        'R2': r2_score(y_true, y_pred)
    }

baseline_metrics = metrics(y_test, baseline_pred)
baseline_metrics


## 5. Polynomial features and interaction terms

`PolynomialFeatures(degree=2, include_bias=False)` creates squared terms and pairwise interaction terms. This lets Linear Regression represent selected non-linear relationships.

In [ ]:
poly_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

poly_model.fit(X_train, y_train)
poly_pred = poly_model.predict(X_test)
poly_metrics = metrics(y_test, poly_pred)
poly_metrics


## 6. Custom mathematical transformations

The distributions of `residual sugar`, `free sulfur dioxide`, and `total sulfur dioxide` are right-skewed. A `log1p(x)` transformation is used because it is defined at zero and reduces the influence of very large values.

In [ ]:
log_features = ['residual sugar', 'free sulfur dioxide', 'total sulfur dioxide']

X_train_eng = X_train.copy()
X_test_eng = X_test.copy()

for col in log_features:
    X_train_eng[col] = np.log1p(X_train_eng[col].clip(lower=0))
    X_test_eng[col] = np.log1p(X_test_eng[col].clip(lower=0))

display(X_train_eng[log_features].describe().T)


## 7. Train the model again on engineered features


In [ ]:
engineered_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

engineered_model.fit(X_train_eng, y_train)
engineered_pred = engineered_model.predict(X_test_eng)
engineered_metrics = metrics(y_test, engineered_pred)
engineered_metrics


## 8. Compare performance


In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Baseline Linear Regression', **baseline_metrics},
    {'Model': 'Polynomial Linear Regression', **poly_metrics},
    {'Model': 'Polynomial + Log Transform', **engineered_metrics}
])
display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=comparison, x='Model', y='RMSE', ax=axes[0])
axes[0].tick_params(axis='x', rotation=25)
axes[0].set_title('RMSE: lower is better')
sns.barplot(data=comparison, x='Model', y='R2', ax=axes[1])
axes[1].tick_params(axis='x', rotation=25)
axes[1].set_title('R²: higher is better')
plt.tight_layout()
plt.show()


## 9. Interpretability and conclusion

- The baseline model provides the benchmark using the original numeric variables.
- Polynomial features add squared and pairwise interaction terms, increasing the model's ability to represent non-linear relationships but also increasing feature count and complexity.
- Log transformation compresses large values in skewed variables and can make relationships easier for a linear model to represent.
- The comparison table should be used to report the actual change in MAE, RMSE and R² after running the notebook.
- A lower RMSE/MAE and higher R² indicate better predictive performance for this regression task. Interpretability should also consider the additional complexity introduced by polynomial terms.


## 10. Reproducibility

Random seed: `42`. The dataset is downloaded directly from the UCI Machine Learning Repository URL used above.